# Fine-tuning + éval compost — version LOCALE (GPU)

Enveloppe du script `scripts/retrain.py`, qui enchaîne tout : split (test compost préservé) →
éval AVANT (pré-entraîné) → fine-tuning → éval APRÈS → comparaison.

**Prérequis** : venv installé, nouvelles annotations intégrées (`update_dataset.py`),
et le pré-entraîné en place : `models/v0_pretrain_yolov8n.pt` (ou rtdetr — voir models/README.md).

**Utilisation** : règle la cellule *Config*, puis *Run All*.

In [ ]:
# --- Config ---
PRETRAIN = "models/v0_pretrain_yolov8n.pt"   # ou models/v0_pretrain_rtdetr-l.pt (voir models/README.md)
EPOCHS   = 50
BATCH    = 8            # baisse à 4 si erreur mémoire GPU
DEPLOY   = False        # True = copie le best.pt vers l'interface à la fin

import os, subprocess
os.chdir(os.path.expanduser("~/stage/Compost_Waste_Yolo"))
PYTHON = os.path.join(os.getcwd(), "venv", "bin", "python")   # le venv unique de la racine
ENV = {**os.environ, "MPLBACKEND": "Agg"}   # le backend inline de Jupyter est invalide hors notebook
print("dossier :", os.getcwd())

## 1. (Optionnel) Intégrer les nouvelles annotations de l'interface

In [ ]:
subprocess.call([PYTHON, "scripts/update_dataset.py"], env=ENV)

## 2. Réentraîner (split → éval avant → fine-tune → éval après)

In [ ]:
cmd = [PYTHON, "scripts/retrain.py", "--pretrain", PRETRAIN,
       "--epochs", str(EPOCHS), "--batch", str(BATCH)] + (["--deploy"] if DEPLOY else [])
if subprocess.call(cmd, env=ENV) != 0:
    raise RuntimeError("échec du réentraînement — voir le journal ci-dessus")

## 3. Visuels de la dernière éval (après fine-tuning)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
evals = sorted(Path("runs").glob("eval_finetune_*"), key=lambda p: p.stat().st_mtime)
assert evals, "aucune éval finetune trouvée"
for name in ("per_class_metrics.png", "confusion_matrices.png"):
    f = evals[-1] / name
    if f.exists(): print(f.name); display(Image(str(f)))